# NIMBO — Real Image Inference Evaluation (Colab T4)

**Objective:** Compare standard SD1.5 denoising against LCM-LoRA accelerated inference on the **same T4 GPU**, using the same prompts/seeds/resolution.

This compares **whole inference pipelines**, not an isolated LoRA-only effect. Different schedulers, guidance scales and step counts are intrinsic to the tested pipelines. Quality is **not proven** by latency; review every paired output and record preferences separately.

Open in Google Colab → Runtime → Change runtime type → **T4 GPU**. If unavailable, stop instead of running on CPU. Model access / terms may vary.

In [ ]:
%pip -q install "diffusers>=0.30,<1" "transformers>=4.44,<5" "accelerate>=0.30" "safetensors>=0.4" "peft>=0.12" pandas matplotlib

In [ ]:
import json, gc, time, platform, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from diffusers import StableDiffusionPipeline, LCMScheduler
from PIL import Image, ImageDraw

assert torch.cuda.is_available(), 'Choose T4 GPU in Colab Runtime settings.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
OUTPUT = Path('nimbo_image_results'); OUTPUT.mkdir(exist_ok=True)
BASE = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
LORA = 'latent-consistency/lcm-lora-sdv1-5'
PROMPTS = [
 'a sunlit modern library with oak bookshelves, architectural photograph',
 'a small red sailboat on a calm lake at dawn, cinematic photography',
 'a minimal white ceramic teapot on a wooden table, studio product photograph',
 'a futuristic clean energy city at sunset, wide angle photography',
]
SEEDS=[17,23,41,59]
WIDTH=512; HEIGHT=512
WARMUP=1; REPEATS=2  # increase REPEATS to 3+ for final measurement

## Measurement protocol
Each mode uses a freshly loaded pipeline to avoid contamination from the other mode. A warmup is excluded. Synchronize CUDA before and after inference. `peak_allocated_gb` is PyTorch peak allocated VRAM during inference; **not** complete GPU process memory or electricity use. Each example saves its image and per-run JSON/CSV metrics. GPU may retain caching allocation across runs.

In [ ]:
def make_pipeline(mode):
    pipe=StableDiffusionPipeline.from_pretrained(BASE, torch_dtype=torch.float16, safety_checker=None)
    pipe=pipe.to('cuda')
    pipe.set_progress_bar_config(disable=True)
    if mode=='lcm':
        pipe.load_lora_weights(LORA)
        pipe.scheduler=LCMScheduler.from_config(pipe.scheduler.config)
    return pipe

def measure(pipe, mode, prompt, seed):
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    start=time.perf_counter()
    generator=torch.Generator(device='cuda').manual_seed(seed)
    result=pipe(prompt, num_inference_steps=25 if mode=='standard' else 4,
                guidance_scale=7.5 if mode=='standard' else 1.0,
                width=WIDTH, height=HEIGHT, generator=generator)
    torch.cuda.synchronize()
    elapsed=time.perf_counter()-start
    peak=torch.cuda.max_memory_allocated()/1e9
    return result.images[0],float(elapsed),float(peak)

rows=[]
for mode in ('standard','lcm'):
    print('Loading',mode)
    pipe=make_pipeline(mode)
    # warmup same pipeline; warmup results not reported
    for _ in range(WARMUP):
        _im,_,_=measure(pipe,mode,PROMPTS[0],SEEDS[0]); del _im
    for i,(prompt,seed) in enumerate(zip(PROMPTS,SEEDS)):
        for repeat in range(REPEATS):
            im,secs,peak=measure(pipe,mode,prompt,seed)
            file=OUTPUT/f'{mode}_prompt{i:02d}_run{repeat:02d}.png'
            im.save(file)
            rows.append({'mode':mode,'prompt_id':i,'seed':seed,'repeat':repeat,
                         'steps':25 if mode=='standard' else 4,
                         'guidance':7.5 if mode=='standard' else 1.0,
                         'latency_s':round(secs,4),'peak_allocated_gb':round(peak,4),
                         'image':str(file),'prompt':prompt})
            print(mode,i,repeat,round(secs,2),'s',round(peak,2),'GB')
    del pipe
    gc.collect();torch.cuda.empty_cache()

df=pd.DataFrame(rows)
df.to_csv(OUTPUT/'per_run.csv',index=False)
print(df.groupby('mode')[['latency_s','peak_allocated_gb']].agg(['mean','median','max']).round(3))

In [ ]:
standard=df[df['mode']=='standard'].latency_s.mean()
fast=df[df['mode']=='lcm'].latency_s.mean()
summary={'gpu':torch.cuda.get_device_name(0),'gpu_total_gb':round(torch.cuda.get_device_properties(0).total_memory/1e9,3),
         'base_model':BASE,'lora':LORA,'width':WIDTH,'height':HEIGHT,'prompt_count':len(PROMPTS),
         'repeats_per_prompt':REPEATS,'warmup_excluded':WARMUP,
         'standard_mean_latency_s':round(float(standard),4),
         'lcm_mean_latency_s':round(float(fast),4),
         'latency_reduction_pct':round(float((1-fast/standard)*100),2),
         'standard_mean_peak_allocated_gb':round(float(df[df['mode']=='standard'].peak_allocated_gb.mean()),3),
         'lcm_mean_peak_allocated_gb':round(float(df[df['mode']=='lcm'].peak_allocated_gb.mean()),3),
         'quality':'NOT YET ASSESSED',
         'limitations':['pipeline comparison; no isolation of LoRA vs schedule/steps',
                        'PyTorch peak allocated VRAM, not system/process VRAM',
                        'No energy/cost measurement','No automated proof of visual quality']}
(OUTPUT/'summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))

## Compare images and record quality
The following shows side-by-side generated images from the first run with matched prompts/seeds. Similar seeds **do not mean pixel-correspondence** across different schedulers. After looking, manually rate prompt adherence and perceptual quality **without seeing the mode label** (blind review recommended for final report).

In [ ]:
from IPython.display import display
for i,prompt in enumerate(PROMPTS):
    left=Image.open(OUTPUT/f'standard_prompt{i:02d}_run00.png')
    right=Image.open(OUTPUT/f'lcm_prompt{i:02d}_run00.png')
    canvas=Image.new('RGB',(WIDTH*2,HEIGHT+36),'white')
    canvas.paste(left,(0,36));canvas.paste(right,(WIDTH,36))
    d=ImageDraw.Draw(canvas);d.text((10,10),'Standard 25 steps',fill='black');d.text((WIDTH+10,10),'LCM-LoRA 4 steps',fill='black')
    display(canvas)
    print('Prompt:',prompt)

**Output files:** `nimbo_image_results/per_run.csv`, `summary.json`, PNG comparisons. In Colab, zip results using `!zip -qr nimbo_image_results.zip nimbo_image_results` then download the ZIP. Do not claim equivalent image quality until blind comparisons are scored.